# Merge phenotype data with polygenic scores and PCs

In [ ]:
output_dir = os.environ.get('WORKSPACE_BUCKET')

## load PCs

In [ ]:
genomic_data_path = "/home/jupyter/workspace/Data_from_All_of_Us_Controlled_tier/vwb-aou-datasets-controlled"
!ls "{genomic_data_path}"

In [ ]:
import pandas as pd
ancestry_pred = pd.read_csv(f"{genomic_data_path}/v8/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv", delimiter="\t")

In [ ]:
PCs = ancestry_pred["pca_features"].str.split(",", n = 16, expand = True)


In [ ]:
pid = ancestry_pred[["research_id", 'ancestry_pred']]
pid.head(5)

In [ ]:
columns= ["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10","PC11","PC12","PC13","PC14","PC15","PC16"]
PCs.columns = columns

In [ ]:
PCs_final = pd.concat([pid, PCs], axis = 1)
PCs_final.head(5)

# load phenotype data

In [ ]:
cohort_polypharmacy_df = pd.read_csv(f"{output_dir}/data/cohort_polypharmacy_with_outcome.csv")
cohort_polypharmacy_df

In [ ]:
pid_with_drugs = set(cohort_polypharmacy_df['person_id'].to_list())

In [ ]:
psam_chr1_path = f"{genomic_data_path}/v8/wgs/short_read/snpindel/acaf_threshold/pgen/acaf_threshold.chr1.psam"
short_read_WGS_psam_df = pd.read_csv(psam_chr1_path, sep='\t')
short_read_WGS_psam_df.head(5)

In [ ]:
srWGS_pids = set(short_read_WGS_psam_df["#IID"].tolist())
len(srWGS_pids)

In [ ]:
len(pid_with_drugs-srWGS_pids)

In [ ]:
len(srWGS_pids - pid_with_drugs)

In [ ]:
cohort_pid = srWGS_pids & pid_with_drugs
len(cohort_pid)

In [ ]:
cohort_polypharmacy_df.columns

In [ ]:
cohort_demographic = cohort_polypharmacy_df[cohort_polypharmacy_df['person_id'].isin(cohort_pid)]
len(cohort_demographic)

In [ ]:
pheno_final = cohort_demographic.merge(PCs_final, left_on = "person_id", right_on = "research_id", how = "left").drop(["research_id"], axis = 1)
pheno_final = pheno_final.drop(columns=["Unnamed: 0"])
pheno_final.head(5)

In [ ]:
pheno_final.insert(0, "FID", pheno_final.person_id)
pheno_final.head(5)

In [ ]:
pheno_final = pheno_final.rename(columns={'person_id': 'IID', "age": "Age", "sex": "Sex"})

In [ ]:
pheno_final[['IID']].to_csv(f"{output_dir}/results/keep_samples.txt", sep=" ", index=False, header=False)
pheno_final.to_csv(f"{output_dir}/results/phenotype_all.csv")